In [1]:
import matplotlib.pyplot as plt
import os
import numpy as np
import torch

print(torch.cuda.is_available())
# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
# 设置文件夹
current_dir = os.getcwd()
data_dir = os.path.join(current_dir, "txt")
namelist = []

def make_file_path(name):
    file_paths = []
    for i in range(1, 6):
        sample_name = "samples_0_" + name + "_" + str(i) + ".txt"
        file_paths.append(os.path.join(data_dir,sample_name))
    return file_paths

def get_datas(name):
    file_paths = make_file_path(name)
    datas = []
    for file_path in file_paths:
        if os.path.exists(file_path):
            x, y = [], []  # Reset x and y for each file
            with open(file_path, 'r') as file:
                for line in file:
                    # 按空格分隔每行数据
                    parts = line.strip().split()
                    if(float(parts[0])>500):
                        break
                    if len(parts) == 2:
                        x.append(float(parts[0]))
                        y.append(float(parts[1]))
            data = np.array([x, y])
            datas.append(data)
        else:
            return False
    namelist.append(name)
    return np.array(datas)

def get_all_patients_datas(cols = None):
    """
    获取所有患者的数据
    返回一个列表，列表中的每个元素是一个字典，字典包含三个键：col, index, datas。
    col: 列名
    index: 行号
    """
    if(cols == None):
        cols = ["A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K", "L",  "P"]
    all_patients_datas = []  # 使用列表存储每个患者的数据
    for col in cols:
        for i in range(1, 30):
            datas = get_datas(col + str(i))
            if datas is not False:
                # 将每个患者的数据存储为字典
                all_patients_datas.append(datas)
            else:
                continue
    return np.array(all_patients_datas)

# 设置小数据集
dcols = ["A"]
origin_patients = get_all_patients_datas()
print(origin_patients.shape)
print(f"获取到 {len(origin_patients)} 个患者的数据。")

False
Using device: cpu
(275, 5, 2, 118340)
获取到 275 个患者的数据。


In [2]:
def draw(datas):
    plt.figure(figsize=(15, 10), dpi=150)  # 调整图像大小和分辨率
    plt.rcParams['lines.linewidth'] = 1.0  # 调整线条宽度
    plt.xlabel("X-axis Label")  # 添加 X 轴标签
    plt.ylabel("Y-axis Label")  # 添加 Y 轴标签
    plt.title("Improved Plot")  # 添加标题
    plt.grid(True)  # 添加网格
    plt.tight_layout()  # 自动调整布局
    for data in datas:
        x, y = data
        plt.plot(x, y)
    plt.show()

In [3]:
from scipy.ndimage import grey_opening, grey_closing

corrected_patients = origin_patients.copy()
def baseline_correct():
    for i, patient_data in enumerate(origin_patients):
        for j, data in enumerate(patient_data):
            x, y = data
            # 使用灰度开运算进行基线校正
            dy = grey_opening(y, size=1000)
            # 使用灰度闭运算进行基线校正
            dy = grey_closing(dy, size=1000)
            corrected_patients[i][j][1] = y - dy  # 更新校正后的数据

baseline_correct()
del origin_patients

In [4]:
def draw_smooth(data1, data2,xlim=None):
    data_noise = np.array([data1[0], data1[1] - data2[1]])
    plt.figure(figsize=(15, 10), dpi=150)  # 调整图像大小和分辨率
    plt.rcParams['lines.linewidth'] = 1.0  # 调整线条宽度
    plt.xlabel("X-axis Label")  # 添加 X 轴标签
    plt.ylabel("Y-axis Label")  # 添加 Y 轴标签
    plt.title("Improved Plot")  # 添加标题
    plt.grid(True)  # 添加网格
    plt.tight_layout()  # 自动调整布局
    plt.plot(data_noise[0], data_noise[1], label="Noise Data", color='red')
    plt.show()

    if xlim is None:
        xlim = (data1[0].min(), data1[0].max())
    mask = (data1[0] > xlim[0]) & (data1[0] < xlim[1])
    plt.figure(figsize=(15, 10), dpi=150)  # 调整图像大小和分辨率
    plt.rcParams['lines.linewidth'] = 1.0  # 调整线条宽度
    plt.xlabel("X-axis Label")  # 添加 X 轴标签
    plt.ylabel("Y-axis Label")  # 添加 Y 轴标签
    plt.title("Improved Plot")  # 添加标题
    plt.grid(True)  # 添加网格
    plt.tight_layout()  # 自动调整布局
    plt.plot(data1[0][mask], data1[1][mask], label="Data 1", color='blue')
    plt.plot(data2[0][mask], data2[1][mask], label="Data 2", color='green')
    plt.show()


In [5]:
from scipy.signal import savgol_filter
smooth_patients = corrected_patients.copy()
mask = smooth_patients[0][0][0]<500
def wavelet_smooth():
    for i, patient_datas in enumerate(smooth_patients):
        for j, data in enumerate(patient_datas):
            x, y = data
            smoothed_y = savgol_filter(y, window_length=11, polyorder=3)
            smooth_patients[i][j][1] = smoothed_y
            smooth_patients[i][j][0] = x
wavelet_smooth()

for i, patient_datas in enumerate(smooth_patients):
    for j, data in enumerate(patient_datas):
        x, y = data
        x = x[mask]
        y = y[mask]
del corrected_patients


In [6]:
import pywt

peaked_patients = smooth_patients.copy()

def peak_extract():
    window_size = 100
    for i, patient in enumerate(peaked_patients):
        for j, data in enumerate(patient):
            x, y = data
            scales = np.arange(1, 20)
            coefficients = np.zeros((len(scales), len(y)))
            for k, scale in enumerate(scales):
                wavelet = pywt.cwt(y, scales=[scale], wavelet='mexh')[0][0]
                coefficients[k] = wavelet
            # 锐化以后的y
            sharpened_y=np.sum(coefficients,axis=0)
            sharpened_y = np.maximum(sharpened_y, 0)
            # if(i%5==0 and j==0):
                # print(f"Patient {i}, Sample {j}: {len(sharpened_y)} points")
                # 绘制锐化后的数据
                # draw_smooth([x, sharpened_y], [x, y], xlim=(300, 350))
            # 计算局部最大值
            peaks = np.where((sharpened_y[1:-1] > sharpened_y[:-2]) & (sharpened_y[1:-1] > sharpened_y[2:]))[0] + 1
            # print(np.count_nonzero(peaks))
            y_filtered = np.zeros_like(y)
            for peak in peaks:
                signal = sharpened_y[peak]
                noise = np.std(sharpened_y[max(0,peak - window_size):min(len(sharpened_y), peak + window_size)])
                snr = signal / noise
                if snr > 3:
                    y_filtered[peak] = y[peak]
            peaked_patients[i][j][1] = y_filtered

peak_extract()
del smooth_patients

In [7]:
def draw_peaks(data1, data2, xlim=None):
    if xlim is None:
        xlim = (data1[0].min(), data1[0].max())
    plt.figure(figsize=(15, 10), dpi=150)  # 调整图像大小和分辨率
    plt.rcParams['lines.linewidth'] = 1.0  # 调整线条宽度
    plt.xlabel("X-axis Label")  # 添加 X 轴标签
    plt.ylabel("Y-axis Label")  # 添加 Y 轴标签
    plt.title("Improved Plot")  # 添加标题
    plt.grid(True)  # 添加网格
    plt.tight_layout()  # 自动调整布局
    x1, y1 = data1[0], data1[1]  # 分离 x 和 y
    x2, y2 = data2[0], data2[1]  # 分离 x 和 y
    mask = (x1 > xlim[0]) & (x1 < xlim[1])
    x1, y1 = x1[mask], y1[mask]  # 正确过滤
    x2, y2 = x2[mask], y2[mask]  # 正确过滤
    plt.scatter(x1, y1, color='red', label='Peaks (y1)', s=10)  # y1 as scatter plot
    plt.plot(x2, y2, color='blue', label='Smoothed Data (y2)')  # y2 as line plot
    plt.legend()  # Add legend
    plt.show()

In [8]:
for i in range(len(namelist)):
    print("numpeaks:", np.count_nonzero(peaked_patients[i][0][1]))

numpeaks: 673
numpeaks: 677
numpeaks: 733
numpeaks: 671
numpeaks: 705
numpeaks: 662
numpeaks: 718
numpeaks: 655
numpeaks: 613
numpeaks: 695
numpeaks: 626
numpeaks: 676
numpeaks: 688
numpeaks: 701
numpeaks: 697
numpeaks: 679
numpeaks: 679
numpeaks: 648
numpeaks: 622
numpeaks: 690
numpeaks: 664
numpeaks: 703
numpeaks: 693
numpeaks: 700
numpeaks: 653
numpeaks: 741
numpeaks: 678
numpeaks: 662
numpeaks: 770
numpeaks: 671
numpeaks: 627
numpeaks: 766
numpeaks: 665
numpeaks: 686
numpeaks: 789
numpeaks: 637
numpeaks: 661
numpeaks: 753
numpeaks: 668
numpeaks: 663
numpeaks: 758
numpeaks: 656
numpeaks: 667
numpeaks: 803
numpeaks: 669
numpeaks: 655
numpeaks: 777
numpeaks: 719
numpeaks: 780
numpeaks: 660
numpeaks: 680
numpeaks: 668
numpeaks: 701
numpeaks: 651
numpeaks: 671
numpeaks: 659
numpeaks: 686
numpeaks: 735
numpeaks: 674
numpeaks: 702
numpeaks: 675
numpeaks: 695
numpeaks: 668
numpeaks: 651
numpeaks: 713
numpeaks: 635
numpeaks: 691
numpeaks: 658
numpeaks: 636
numpeaks: 689
numpeaks: 641
numpea

In [9]:
import numpy as np
from scipy.optimize import curve_fit

def multi_gauss(x, *params):
    """
    多高斯函数模型
    :param params: 每3个参数表示一个高斯峰 (A, mu, sigma)
    """
    y = np.zeros_like(x)
    for i in range(0, len(params), 3):
        A, mu, sigma = params[i:i+3]
        y += A * np.exp(-(x - mu)**2 / (2 * sigma**2))
    return y

In [10]:
from scipy.signal import find_peaks

def detect_initial_params(x, y, n_peaks=5):
    """
    检测初始高斯参数
    """
    peaks, _ = find_peaks(y, height=np.median(y[y>0]), distance=5)  # 调整distance避免过近峰
    peaks = peaks[:n_peaks]  # 限制最大峰数量
    
    initial_params = []
    for peak in peaks:
        A = y[peak]               # 幅度
        mu = x[peak]              # 中点
        sigma = 0.5 * (x[1] - x[0]) * 5  # 初始sigma（根据x轴间隔估算）
        initial_params.extend([A, mu, sigma])
    
    return initial_params

In [11]:
def fit_multi_gauss(x, y, max_peaks=5):
    """
    拟合多高斯模型
    """
    # 检测初始参数
    initial_params = detect_initial_params(x, y, n_peaks=max_peaks)
    
    # 设置参数边界（避免sigma过大或过小）
    n_params = len(initial_params)
    bounds = (
        [0, -np.inf, 1e-6] * (n_params // 3),  # 下界
        [np.inf, np.inf, 10] * (n_params // 3)   # 上界
    )
    
    # 拟合
    popt, pcov = curve_fit(multi_gauss, x, y, p0=initial_params, bounds=bounds)
    
    return popt

In [12]:
from scipy.optimize import curve_fit

sum_y = np.zeros_like(peaked_patients[0][0][1])
for i in range(len(peaked_patients)):
    for j in range(len(peaked_patients[i])):
        sum_y += peaked_patients[i][j][1]
sum_y = sum_y / len(peaked_patients)/5
# Find the indices of the 1000 largest peaks
largest_peak_indices = np.argpartition(sum_y, -1000)[-1000:]
# Set all other values in sum_y to 0
filtered_sum_y = np.zeros_like(sum_y)
filtered_sum_y[largest_peak_indices] = sum_y[largest_peak_indices]
# sum_y = filtered_sum_y

# plt.figure(figsize=(15, 10), dpi=150)  # 调整图像大小和分辨率
# plt.rcParams['lines.linewidth'] = 1.0  # 调整线条宽度
# plt.xlabel("X-axis Label")  # 添加 X 轴标签
# plt.ylabel("Y-axis Label")  # 添加 Y 轴标签
# # plt.xlim(215,220)
# plt.title("Average Peaks")  # 添加标题
# plt.grid(True)  # 添加网格
# plt.tight_layout()  # 自动调整布局
# plt.plot(peaked_patients[0][0][0], sum_y, label="Average Peaks", color='blue')
# plt.scatter(peaked_patients[0][0][0], filtered_sum_y, color='red', label='Filtered Peaks', s=10)  # 绘制过滤后的峰值
# plt.legend()  # 添加图例
# plt.show()



In [13]:
# 原始数据
x = peaked_patients[0][0][0]  # m/z轴
y = filtered_sum_y            # 强度

# 拟合多高斯模型（假设最多5个峰）
popt = fit_multi_gauss(x, y, max_peaks=500)

# 生成拟合曲线
fit_y = multi_gauss(x, *popt)

# 提取每个峰的参数（A, mu, sigma）
n_peaks = len(popt) // 3
peaks_info = []
for i in range(n_peaks):
    A, mu, sigma = popt[i*3:(i+1)*3]
    area = A * sigma * np.sqrt(2 * np.pi)  # 高斯峰面积公式
    peaks_info.append({"mu": mu, "area": area, "sigma": sigma})



In [14]:
new_y = np.zeros_like(y)
for i in range(len(peaks_info)):
    A = peaks_info[i]["area"]
    mu = peaks_info[i]["mu"]
    sigma = peaks_info[i]["sigma"]
    idx_mu = np.argmin(np.abs(x - mu))  # 找到最接近mu的索引
    new_y[idx_mu] =A

In [15]:
# plt.figure(figsize=(15, 6))
# plt.plot(x, y, 'b-', label='Original Peaks')
# plt.plot(x, fit_y, 'r--', label='Multi-Gauss Fit')
# plt.plot(x, new_y, 'g-', label='Replaced Peaks')
# plt.scatter([peak["mu"] for peak in peaks_info], 
#             [peak["area"] for peak in peaks_info], 
#             color='green', label='Fitted Peaks (Center)')
# # plt.xlim(320, 322)
# plt.xlabel("m/z")
# plt.ylabel("Intensity")
# plt.title("Multi-Gaussian Decomposition")
# plt.legend()
# plt.grid(True)
# plt.show()

In [16]:
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter

def dynamic_local_align(x, y, ref_y, window_size=10,max_shift=5,cor_bar = 0.2):
    """
    动态局部对齐：分段计算偏移量并平滑过渡
    :param window_size: 分段窗口大小（m/z单位）
    :param overlap: 相邻窗口重叠区大小（确保平滑过渡）
    """
    # 初始化输出
    y_aligned = np.zeros_like(y)
    peaks = np.nonzero(y)[0]  # 获取非零峰值索引
    # print(f"Number of peaks: {len(peaks)}")
    for peak in peaks:
        y_patch = y[max(0, peak - window_size):min(len(y), peak + window_size)]
        start = max(0, peak - window_size)
        end = min(len(y), peak + window_size)
        best_shift = 0
        max_corr = 0
        for shift in range(-max_shift, max_shift + 1):
            if(start + shift < 0 or end+shift >= len(y)):
                continue
            ref_y_patch = ref_y[start + shift:end + shift]
            cor = np.corrcoef(y_patch, ref_y_patch)[0, 1]
            if cor > max_corr and cor > cor_bar:
                max_corr = cor
                best_shift = shift
        y_aligned[peak+ best_shift] = y[peak]  # 将峰值对齐到参考位置
    return y_aligned

# 对所有样本应用动态对齐
aligned_patients = []
for i, patient in enumerate(peaked_patients):
    aligned_samples = []
    for j, (x, y) in enumerate(patient):
        y_aligned = dynamic_local_align(x, y, fit_y, window_size=5, max_shift=30, cor_bar=0.1)
        aligned_samples.append((x, y_aligned))
    aligned_patients.append(aligned_samples)

c:\Users\86183\miniconda3\envs\zixian\Lib\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\86183\miniconda3\envs\zixian\Lib\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\86183\miniconda3\envs\zixian\Lib\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: divide by zero encountered in divide
  c /= stddev[:, None]
c:\Users\86183\miniconda3\envs\zixian\Lib\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: divide by zero encountered in divide
  c /= stddev[None, :]


In [17]:
# import numpy as np
# import pandas as pd
# import seaborn as sns
# import matplotlib.pyplot as plt

# def get_data_matrix(patients):
#     """将患者数据转换为矩阵 (n_samples, n_mz_points)"""
#     y_matrix = []
#     for patient in patients:
#         for x, y in patient:
#             y_matrix.append(y)
#     return np.vstack(y_matrix)

# # 获取原始数据和对齐后的数据矩阵
# original_matrix = get_data_matrix(peaked_patients)  # 形状: (n_samples, n_points)
# aligned_matrix = get_data_matrix(aligned_patients)  # 形状: (n_samples, n_points)

# print(f"Original matrix shape: {original_matrix.shape}")
# print(f"Aligned matrix shape: {aligned_matrix.shape}")

In [18]:
# # 计算原始数据和对齐数据的样本间相关系数
# original_corr = np.corrcoef(original_matrix)  # 形状: (n_samples, n_samples)
# aligned_corr = np.corrcoef(aligned_matrix)    # 形状: (n_samples, n_samples)
# # 设置绘图风格
# # plt.style.use('seaborn')
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# # 原始数据热图
# sns.heatmap(
#     original_corr,
#     ax=ax1,
#     cmap='coolwarm',
#     vmin=0,
#     vmax=1,
#     square=True,
#     cbar_kws={'label': 'Correlation'}
# )
# ax1.set_title('Original Data Correlation', fontsize=14)
# ax1.set_xlabel('Sample Index')
# ax1.set_ylabel('Sample Index')

# # 对齐数据热图
# sns.heatmap(
#     aligned_corr,
#     ax=ax2,
#     cmap='coolwarm',
#     vmin=0,
#     vmax=1,
#     square=True,
#     cbar_kws={'label': 'Correlation'}
# )
# ax2.set_title('Aligned Data Correlation', fontsize=14)
# ax2.set_xlabel('Sample Index')
# ax2.set_ylabel('Sample Index')

# plt.tight_layout()
# plt.show()

In [19]:
import pandas as pd
import numpy as np

# 计算均值数据 (n_patients, 2, n_points)
mean_matched = np.mean(aligned_patients, axis=1)  

# 计算所有患者y值的总和，并筛选Top 800峰
sum_y = np.sum([data[1] for data in mean_matched], axis=0)
top800_indices = np.argpartition(sum_y, -800)[-800:]  # 获取前800大峰的索引
cols_mask = np.zeros_like(sum_y, dtype=bool)
cols_mask[top800_indices] = True  # 生成布尔掩码

# 提取过滤后的m/z值和对应y值
filtered_x = mean_matched[0][0][cols_mask]

# 构建数据字典
data = {
    "Sample_Name": namelist,
    **{f"m/z_{i:.3f}": [] for i in filtered_x}  # 保留5位小数确保唯一性
}

# 填充每个样本的y值
for x, y in mean_matched:
    y_filtered = y[cols_mask]
    for j, mz in enumerate(filtered_x):
        data[f"m/z_{mz:.3f}"].append(y_filtered[j])

# 转换为DataFrame
df = pd.DataFrame(data)

# 新增两列 ---------------------------------------------------
# 1. 计算每行的非零值数量（跳过Sample_Name列）
df['nonzeros'] = df.iloc[:, 1:].astype(bool).sum(axis=1)

# 2. 计算每行的总强度（跳过Sample_Name列）
df['total_intensity'] = df.iloc[:, 1:-1].sum(axis=1)  # -1是为了排除刚添加的nonzeros列

# 列顺序调整（可选）
cols = ['Sample_Name', 'nonzeros', 'total_intensity'] + [c for c in df.columns if c.startswith('m/z_')]
df = df[cols]

# 保存为CSV
df.to_csv("mean_matched_top800_with_stats.csv", index=False)

print(f"已保存包含统计列的数据，形状: {df.shape}")
print(df[['Sample_Name', 'nonzeros', 'total_intensity']].head())  # 预览新增列

已保存包含统计列的数据，形状: (275, 803)
  Sample_Name  nonzeros  total_intensity
0          A1        80     1.352047e+06
1          A2        89     6.936263e+05
2          A3        75     3.039842e+05
3          A4        83     1.225001e+06
4          A5        86     4.013162e+05
